# LeetCode 1980 — Find Unique Binary String bằng Python

Link đề: https://leetcode.com/problems/find-unique-binary-string/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, String, Backtracking

Skeleton LeetCode:

```python
class Solution:
    def findDifferentBinaryString(self, nums: List[str]) -> str:
        ...
```

Bài có **nhiều đáp án** — mọi test dùng hàm kiểm tra hợp lệ. Notebook đi từ **duyệt mọi xâu nhị phân độ dài `n`** → **chỉ cần thử `n + 1` số đầu tiên** (nguyên lý chuồng bồ câu) → **đếm số bit 1** (có `n + 1` giá trị, chỉ `n` xâu) → **đường chéo Cantor** `O(n)` → **quay lui trên trie** (tổng quát khi số xâu bất kỳ).

## 1. Mục tiêu học

1. Áp dụng **nguyên lý chuồng bồ câu**: `n` xâu không thể phủ hết `n + 1` "chuồng" ⇒ luôn có chuồng trống.
2. Chuyển đổi xâu nhị phân ↔ số nguyên bằng `int(s, 2)` và `format(x, f'0{n}b')`.
3. Hiểu **phép chéo hóa Cantor**: xâu `ans[i] = đảo(nums[i][i])` khác mọi `nums[i]` ở vị trí `i`.
4. Viết hàm kiểm tra cho bài nhiều đáp án và dùng nó trong test.
5. Mở rộng: tìm xâu thiếu khi số xâu `m` khác `n` (quay lui có cắt nhánh trên trie).

## 2. Đề bài & ví dụ

Cho mảng `nums` gồm `n` xâu nhị phân **đôi một khác nhau**, mỗi xâu dài đúng `n`. Trả về **một** xâu nhị phân dài `n` **không** xuất hiện trong `nums`. Có nhiều đáp án thì trả bất kỳ.

**Ví dụ 1:** `nums = ["01","10"]` → `"11"` (hoặc `"00"`).

**Ví dụ 2:** `nums = ["00","01"]` → `"11"` (hoặc `"10"`).

**Ví dụ 3:** `nums = ["111","011","001"]` → `"101"` (hoặc `"000"`, `"010"`, `"100"`, `"110"`).

**Ràng buộc:** `n == nums.length`, `1 ≤ n ≤ 16`, `nums[i].length == n`, chỉ gồm `'0'`/`'1'`, các xâu khác nhau.

In [1]:
import random
import time
from itertools import product
from typing import List

examples = [["01", "10"], ["00", "01"], ["111", "011", "001"]]

def is_valid(nums, ans):
    n = len(nums)
    return isinstance(ans, str) and len(ans) == n and set(ans) <= {'0', '1'} and ans not in set(nums)

for nums in examples:
    n = len(nums)
    all_missing = [''.join(t) for t in product('01', repeat=n) if ''.join(t) not in nums]
    print(f"{nums}: mọi đáp án hợp lệ = {all_missing}")

['01', '10']: mọi đáp án hợp lệ = ['00', '11']
['00', '01']: mọi đáp án hợp lệ = ['10', '11']
['111', '011', '001']: mọi đáp án hợp lệ = ['000', '010', '100', '101', '110']


## 3. Kiến thức nền

### 3.1 Xâu nhị phân ↔ số nguyên

In [2]:
print(int("101", 2), format(5, '03b'), format(1, '04b'), bin(5))

5 101 0001 0b101


### 3.2 `set` để kiểm tra "đã có chưa" trong `O(1)` (trung bình)

In [3]:
seen = {"01", "10"}
print("11" in seen, "01" in seen)

False True


### 3.3 Nguyên lý chuồng bồ câu

Nếu có `k` chuồng và **ít hơn `k`** con bồ câu thì ít nhất một chuồng **trống**. Ở đây: `n` xâu, nhưng ta có thể chọn ra `n + 1` "chuồng" (ví dụ các số `0..n`, hoặc số bit 1 `0..n`) ⇒ luôn có ứng viên chưa bị dùng.

In [4]:
nums = ["111", "011", "001"]
counts = {s.count('1') for s in nums}
print("các số bit 1 đã dùng:", sorted(counts), "| có thể là 0..3 -> trống:", [c for c in range(4) if c not in counts])

các số bit 1 đã dùng: [1, 2, 3] | có thể là 0..3 -> trống: [0]


### 3.4 Đảo một bit ký tự

`'1' if c == '0' else '0'`, hoặc `str(1 - int(c))`, hoặc dùng bảng `{'0': '1', '1': '0'}`.

In [5]:
FLIP = {'0': '1', '1': '0'}
print(''.join(FLIP[c] for c in "0110"))

1001


## 4. Ý tưởng nền tảng / trực giác

Có `2ⁿ` xâu độ dài `n` nhưng chỉ `n` xâu bị "chiếm" — dư rất nhiều (với `n = 16`: `65536` so với `16`). Câu hỏi chỉ là tìm **nhanh** một xâu trống.

**Chéo hóa:** viết `nums` thành bảng `n × n`. Lấy đường chéo `nums[0][0], nums[1][1], …` rồi **đảo** từng bit ⇒ xâu mới khác `nums[i]` ở **vị trí `i`** ⇒ khác **mọi** xâu.

Tính tay ví dụ 3:
```
nums[0] = 1 1 1   -> đường chéo nums[0][0] = 1 -> đảo 0
nums[1] = 0 1 1   -> nums[1][1] = 1 -> đảo 0
nums[2] = 0 0 1   -> nums[2][2] = 1 -> đảo 0
```
⇒ `"000"` — không có trong `nums` ✔ (đề cũng chấp nhận `"000"`).

In [6]:
nums = ["111", "011", "001"]
diag = ''.join(nums[i][i] for i in range(len(nums)))
ans = ''.join(FLIP[c] for c in diag)
print("đường chéo:", diag, "-> đảo:", ans, "| hợp lệ:", is_valid(nums, ans))
for i, s in enumerate(nums):
    print(f"  {ans} khác {s} tại vị trí {i}: {ans[i]} != {s[i]}")

đường chéo: 111 -> đảo: 000 | hợp lệ: True
  000 khác 111 tại vị trí 0: 0 != 1
  000 khác 011 tại vị trí 1: 0 != 1
  000 khác 001 tại vị trí 2: 0 != 1


Hình dung bảng `n × n` với đường chéo được đánh dấu `[ ]` — mỗi hàng "bị loại" bởi đúng một ô trên đường chéo:

In [7]:
def show_diag(nums):
    n = len(nums)
    for i, s in enumerate(nums):
        print("  " + " ".join(f"[{c}]" if j == i else f" {c} " for j, c in enumerate(s)))
    ans = ''.join('1' if nums[i][i] == '0' else '0' for i in range(n))
    print("  đảo đường chéo ->", ans)

show_diag(["1101", "0011", "1110", "0000"])

  [1]  1   0   1 
   0  [0]  1   1 
   1   1  [1]  0 
   0   0   0  [0]
  đảo đường chéo -> 0101


## 5. Cách 1 — Brute Force / baseline: duyệt mọi xâu theo thứ tự

In [8]:
def find_brute(nums):
    n = len(nums)
    seen = set(nums)
    for t in product('01', repeat=n):
        s = ''.join(t)
        if s not in seen:
            return s

for nums in examples:
    ans = find_brute(nums)
    print(nums, "->", ans, is_valid(nums, ans))

['01', '10'] -> 00 True
['00', '01'] -> 10 True
['111', '011', '001'] -> 000 True


**Độ phức tạp:** xấu nhất duyệt `n + 1` xâu đầu tiên trước khi thấy chỗ trống (chỉ `n` xâu bị chiếm!) — nhưng **không dừng sớm** thì lý thuyết là `O(2ⁿ · n)`. Thực ra vòng lặp **luôn dừng** trong `n + 1` bước đầu (chuồng bồ câu) ⇒ `O(n²)`. Đây là nhận xét quan trọng dẫn tới cách 2.

## 6. Ý tưởng thuật toán / bước đột phá

Ba cách "chọn `n + 1` chuồng":
1. **Số nguyên `0..n`:** đổi mỗi xâu thành số; một trong `0, 1, …, n` không xuất hiện.
2. **Số bit 1 `0..n`:** mỗi xâu có số bit 1 trong `[0, n]` — `n + 1` giá trị, chỉ `n` xâu ⇒ có giá trị `c` không ai dùng ⇒ xâu `'1'·c + '0'·(n−c)` chắc chắn không có trong `nums`.
3. **Chéo hóa Cantor:** không cần "chuồng" — dựng trực tiếp xâu khác mọi `nums[i]` ở vị trí `i`. `O(n)`, không cần bộ nhớ phụ.

**Quy tắc chéo hóa:** **nếu** `ans[i] ≠ nums[i][i]` với mọi `i` **thì** `ans ≠ nums[i]` với mọi `i`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Thử các số `0..n`

In [9]:
def find_small_numbers(nums):
    n = len(nums)
    seen = {int(s, 2) for s in nums}
    for x in range(n + 1):                    # n + 1 ứng viên, n xâu => chắc chắn có ứng viên trống
        if x not in seen:
            return format(x, f'0{n}b')

for nums in examples:
    ans = find_small_numbers(nums)
    print(nums, "->", ans, is_valid(nums, ans))

['01', '10'] -> 00 True
['00', '01'] -> 10 True
['111', '011', '001'] -> 000 True


Cần `n + 1 ≤ 2ⁿ` để mọi ứng viên có biểu diễn `n` bit — đúng với mọi `n ≥ 1`. **Độ phức tạp:** `O(n²)` để đổi `n` xâu thành số (mỗi xâu `O(n)`), `O(n)` bộ nhớ.

### 7.2 Cách 3 — Đếm số bit 1

In [10]:
def find_by_popcount(nums):
    n = len(nums)
    used = {s.count('1') for s in nums}
    for c in range(n + 1):
        if c not in used:
            return '1' * c + '0' * (n - c)

for nums in examples:
    ans = find_by_popcount(nums)
    print(nums, "->", ans, is_valid(nums, ans))

['01', '10'] -> 00 True
['00', '01'] -> 11 True
['111', '011', '001'] -> 000 True


**Độ phức tạp:** `O(n²)` (đếm bit), `O(n)` bộ nhớ.

### 7.3 Cách 4 — Đường chéo Cantor (lời giải chính)

In [11]:
def find_cantor(nums):
    return ''.join('1' if nums[i][i] == '0' else '0' for i in range(len(nums)))

for nums in examples:
    ans = find_cantor(nums)
    print(nums, "->", ans, is_valid(nums, ans))

['01', '10'] -> 11 True
['00', '01'] -> 10 True
['111', '011', '001'] -> 000 True


**Độ phức tạp:** `O(n)` thời gian, `O(n)` cho xâu kết quả, không cần `set`.

### 7.4 Cách 5 — Quay lui trên trie (khi số xâu `m` bất kỳ)

Nếu đề tổng quát hơn — `m` xâu độ dài `n` (`m < 2ⁿ`), chéo hóa không dùng được khi `m > n`. Ta dựng **trie** và DFS: ở mỗi nút, đi vào nhánh có **ít** xâu hơn "dung lượng" (`số xâu < 2^(độ sâu còn lại)`) — chắc chắn có lá trống phía dưới.

In [12]:
def find_trie(nums, n=None):
    n = n if n is not None else len(nums)
    cnt = {}                                         # cnt[tiền tố] = số xâu có tiền tố đó
    for s in nums:
        for k in range(n + 1):
            cnt[s[:k]] = cnt.get(s[:k], 0) + 1
    prefix = ""
    for depth in range(n):
        cap = 2 ** (n - depth - 1)                   # số xâu tối đa dưới mỗi nhánh con
        for bit in "01":
            if cnt.get(prefix + bit, 0) < cap:
                prefix += bit
                break
    return prefix

for nums in examples:
    ans = find_trie(nums)
    print(nums, "->", ans, is_valid(nums, ans))
many = [format(x, '04b') for x in range(16) if x != 9]       # 15 xâu độ dài 4, chỉ thiếu 1001
print("15 xâu độ dài 4 ->", find_trie(many, 4))

['01', '10'] -> 00 True
['00', '01'] -> 10 True
['111', '011', '001'] -> 000 True
15 xâu độ dài 4 -> 1001


**Độ phức tạp:** `O(m · n)` dựng bảng tiền tố + `O(n)` đi xuống. Bất biến: tại độ sâu `d`, số xâu có tiền tố hiện tại `< 2^(n−d)` (còn chỗ trống) ⇒ ít nhất một nhánh con có số xâu `< 2^(n−d−1)`.

### 7.5 Cách 6 — Ngẫu nhiên có kiểm tra (Las Vegas)

Sinh ngẫu nhiên một xâu dài `n`, nếu trùng thì sinh lại. Xác suất trùng mỗi lần là `n / 2ⁿ` ⇒ số lần thử kỳ vọng `1 / (1 − n/2ⁿ)` — với `n = 1, 2` là 2 lần, với `n ≥ 5` gần như 1 lần. Luôn đúng khi dừng, nhưng thời gian là ngẫu nhiên.

In [13]:
def expected_tries(n):
    return 1 / (1 - n / 2 ** n)

rnd = random.Random(6)
for n in [1, 2, 3, 4, 8, 16]:
    nums = [format(x, f'0{n}b') for x in rnd.sample(range(2 ** n), n)]
    seen = set(nums); tries_total = 0
    for _ in range(2000):
        t = 0
        while True:
            t += 1
            s_ = ''.join(rnd.choice('01') for _ in range(n))
            if s_ not in seen:
                break
        tries_total += t
    print(f"n={n:>2}: số lần thử trung bình = {tries_total / 2000:.3f} (lý thuyết {expected_tries(n):.3f})")

n= 1: số lần thử trung bình = 2.057 (lý thuyết 2.000)
n= 2: số lần thử trung bình = 1.958 (lý thuyết 2.000)
n= 3: số lần thử trung bình = 1.617 (lý thuyết 1.600)
n= 4: số lần thử trung bình = 1.321 (lý thuyết 1.333)
n= 8: số lần thử trung bình = 1.034 (lý thuyết 1.032)
n=16: số lần thử trung bình = 1.000 (lý thuyết 1.000)


## 8. TRACE từng bước

Trace cách 4 và cách 3 trên ví dụ 1 và 3:

In [14]:
def trace_cantor(nums):
    n = len(nums); out = []
    for i in range(n):
        c = nums[i][i]; f = '1' if c == '0' else '0'; out.append(f)
        print(f"  i={i}: nums[{i}] = {nums[i]}, ký tự chéo '{c}' -> đảo '{f}' | ans đang = {''.join(out)}")
    ans = ''.join(out)
    print("  =>", ans, "| hợp lệ:", is_valid(nums, ans))
    return ans

def trace_popcount(nums):
    n = len(nums)
    used = sorted({s.count('1') for s in nums})
    print(f"  số bit 1 đã dùng: {used} trên {list(range(n + 1))}")
    c = next(c for c in range(n + 1) if c not in used)
    ans = '1' * c + '0' * (n - c)
    print(f"  chọn c = {c} -> {ans} | hợp lệ: {is_valid(nums, ans)}")
    return ans

for nums in (examples[0], examples[2]):
    print(nums, "— Cantor:"); trace_cantor(nums)
    print(nums, "— đếm bit:"); trace_popcount(nums)

['01', '10'] — Cantor:
  i=0: nums[0] = 01, ký tự chéo '0' -> đảo '1' | ans đang = 1
  i=1: nums[1] = 10, ký tự chéo '0' -> đảo '1' | ans đang = 11
  => 11 | hợp lệ: True
['01', '10'] — đếm bit:
  số bit 1 đã dùng: [1] trên [0, 1, 2]
  chọn c = 0 -> 00 | hợp lệ: True
['111', '011', '001'] — Cantor:
  i=0: nums[0] = 111, ký tự chéo '1' -> đảo '0' | ans đang = 0
  i=1: nums[1] = 011, ký tự chéo '1' -> đảo '0' | ans đang = 00
  i=2: nums[2] = 001, ký tự chéo '1' -> đảo '0' | ans đang = 000
  => 000 | hợp lệ: True
['111', '011', '001'] — đếm bit:
  số bit 1 đã dùng: [1, 2, 3] trên [0, 1, 2, 3]
  chọn c = 0 -> 000 | hợp lệ: True


| i | nums[i] | nums[i][i] | ans[i] |
|---|---|---|---|
| 0 | `111` | 1 | 0 |
| 1 | `011` | 1 | 0 |
| 2 | `001` | 1 | 0 |

## 9. Code nộp LeetCode

In [15]:
class Solution:
    def findDifferentBinaryString(self, nums: List[str]) -> str:
        # chéo hóa Cantor: khác nums[i] ở vị trí i
        return ''.join('1' if nums[i][i] == '0' else '0' for i in range(len(nums)))

for nums in examples:
    assert is_valid(nums, Solution().findDifferentBinaryString(nums))
print("Ví dụ đề: OK (kiểm bằng hàm hợp lệ)")

Ví dụ đề: OK (kiểm bằng hàm hợp lệ)


## 10. Vì sao thuật toán đúng

**Cantor.** Xâu `ans` có `ans[i] ≠ nums[i][i]` với mọi `i`. Nếu `ans = nums[j]` với `j` nào đó, thì `ans[j] = nums[j][j]` — mâu thuẫn. Độ dài `ans` là `n` (vì có `n` xâu, mỗi xâu dài `n`, đường chéo luôn tồn tại).

**Số `0..n`.** Có `n + 1` số phân biệt `0..n` (đều biểu diễn được bằng `n` bit vì `n < 2ⁿ`), `n` xâu chiếm tối đa `n` số ⇒ có số trống.

**Số bit 1.** Hàm `s ↦ count('1')` nhận giá trị trong `{0, …, n}` (`n + 1` giá trị); `n` xâu ⇒ có giá trị `c` không ai có. Xâu `1^c 0^(n−c)` có đúng `c` bit 1 ⇒ không thể là bất kỳ `nums[i]` nào.

**Trie.** Bất biến "còn chỗ dưới tiền tố hiện tại" được giữ, nên tới độ sâu `n` có `cnt[prefix] < 1` ⇒ `prefix` không có trong `nums`.

Kiểm chứng bằng máy tính chất then chốt của chéo hóa: với nhiều bộ ngẫu nhiên, `ans` khác `nums[i]` **đúng tại vị trí `i`** (có thể khác thêm ở chỗ khác).

In [16]:
rnd = random.Random(10)
for _ in range(2000):
    n = rnd.randint(1, 16)
    t = [format(x, f'0{n}b') for x in rnd.sample(range(2 ** n), n)]
    a = find_cantor(t)
    assert all(a[i] != t[i][i] for i in range(n))
print("2000 bộ: ans[i] != nums[i][i] với mọi i: OK")

2000 bộ: ans[i] != nums[i][i] với mọi i: OK


## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Duyệt theo thứ tự | `O(n²)` (dừng trong `n + 1` bước) | `O(n²)` (set) |
| 2. Số `0..n` | `O(n²)` | `O(n)` |
| 3. Đếm bit 1 | `O(n²)` | `O(n)` |
| 4. Cantor | `O(n)` | `O(1)` phụ |
| 5. Trie | `O(m · n)` | `O(m · n)` |

In [17]:
rnd = random.Random(1980)
cases = []
for _ in range(1500):
    n = rnd.randint(1, 16)
    cases.append([format(x, f'0{n}b') for x in rnd.sample(range(2 ** n), n)])
for name, fn in [("brute", find_brute), ("small_numbers", find_small_numbers), ("popcount", find_by_popcount), ("cantor", find_cantor), ("trie", find_trie), ("Solution", Solution().findDifferentBinaryString)]:
    t0 = time.perf_counter(); ok = all(is_valid(c, fn(c)) for c in cases); dt = time.perf_counter() - t0
    print(f"{name:<14}: tất cả hợp lệ = {ok}, {dt * 1000:7.1f} ms cho {len(cases)} bộ")

brute         : tất cả hợp lệ = True,     2.0 ms cho 1500 bộ
small_numbers : tất cả hợp lệ = True,     2.4 ms cho 1500 bộ
popcount      : tất cả hợp lệ = True,     2.1 ms cho 1500 bộ
cantor        : tất cả hợp lệ = True,     1.8 ms cho 1500 bộ
trie          : tất cả hợp lệ = True,    20.5 ms cho 1500 bộ
Solution      : tất cả hợp lệ = True,     1.7 ms cho 1500 bộ


## 12. Bộ test

In [18]:
ALL = {
    "brute": find_brute,
    "small_numbers": find_small_numbers,
    "popcount": find_by_popcount,
    "cantor": find_cantor,
    "trie": find_trie,
    "Solution": lambda a: Solution().findDifferentBinaryString(a),
}

def check(nums):
    snap = list(nums)
    for name, fn in ALL.items():
        ans = fn(nums)
        assert is_valid(nums, ans), (name, nums, ans)
        assert nums == snap
    return True

for nums in examples:
    check(nums)
print("Ví dụ đề: OK")

check(["0"]); check(["1"])                                   # n = 1
check(["00", "11"]); check(["10", "01"])
check([format(x, '016b') for x in range(16)])                 # 16 số nhỏ nhất
check(['1' * c + '0' * (16 - c) for c in range(16)])          # chiếm 16 trong 17 giá trị số bit 1
check([format((1 << 16) - 1 - x, '016b') for x in range(16)]) # 16 số lớn nhất
print("Edge case: OK")

# Toàn bộ mọi bộ nums với n <= 3
from itertools import combinations
cnt = 0
for n in range(1, 4):
    allstr = [''.join(t) for t in product('01', repeat=n)]
    for comb in combinations(allstr, n):
        for perm_seed in range(2):
            arr = list(comb); random.Random(perm_seed).shuffle(arr)
            check(arr); cnt += 1
print(f"{cnt} bộ nums (mọi tổ hợp với n <= 3, hai thứ tự): OK")

Ví dụ đề: OK
Edge case: OK
128 bộ nums (mọi tổ hợp với n <= 3, hai thứ tự): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên đệm số 0 khi đổi số → xâu

In [19]:
def find_bug_no_pad(nums):
    seen = {int(s, 2) for s in nums}
    for x in range(len(nums) + 1):
        if x not in seen:
            return bin(x)[2:]                     # SAI: thiếu số 0 đứng đầu

nums = ["111", "011", "001"]
ans = find_bug_no_pad(nums)
print("không đệm:", repr(ans), "hợp lệ?", is_valid(nums, ans), "| đúng:", repr(find_small_numbers(nums)))

không đệm: '0' hợp lệ? False | đúng: '000'


**Vì sao sai:** đáp án phải dài **đúng `n`**. `bin(0)[2:] = "0"` chỉ dài 1. **Sửa:** `format(x, f'0{n}b')`.

### Lỗi 2 — Chỉ thử `n` ứng viên (thiếu một "chuồng")

In [20]:
def find_bug_n_candidates(nums):
    n = len(nums)
    seen = {int(s, 2) for s in nums}
    for x in range(n):                            # SAI: chỉ n ứng viên, có thể bị chiếm hết
        if x not in seen:
            return format(x, f'0{n}b')
    return None

nums = ["00", "01"]
print("thử n số:", find_bug_n_candidates(nums), "| đúng:", find_small_numbers(nums))

thử n số: None | đúng: 10


**Vì sao sai:** `n` xâu có thể chiếm đúng `n` số `0..n−1` ⇒ không tìm được ⇒ trả `None`. **Sửa:** `range(n + 1)`.

### Lỗi 3 — Chéo hóa nhưng **không đảo** bit (lấy nguyên đường chéo)

In [21]:
def find_bug_no_flip(nums):
    return ''.join(nums[i][i] for i in range(len(nums)))      # SAI: có thể trùng một xâu

for t in (["01", "10"], ["10", "11"]):
    ans = find_bug_no_flip(t)
    print(t, "-> đường chéo không đảo:", ans, "hợp lệ?", is_valid(t, ans))

['01', '10'] -> đường chéo không đảo: 00 hợp lệ? True
['10', '11'] -> đường chéo không đảo: 11 hợp lệ? False


**Vì sao sai:** đường chéo **không đảo** có thể trùng chính một xâu trong `nums`: với `["10", "11"]` đường chéo là `"11"` = `nums[1]`. Với `["01", "10"]` thì may mắn ra `"00"` hợp lệ — nhưng không có bảo đảm. Ô sau tìm phản ví dụ một cách hệ thống:

In [22]:
bad = []
for n in range(1, 4):
    allstr = [''.join(t) for t in product('01', repeat=n)]
    for comb in combinations(allstr, n):
        for perm in [list(comb), list(reversed(comb))]:
            if not is_valid(perm, find_bug_no_flip(perm)):
                bad.append(perm)
print(f"{len(bad)} bộ làm bản 'không đảo' sai, ví dụ: {bad[:3]}")

72 bộ làm bản 'không đảo' sai, ví dụ: [['0'], ['0'], ['1']]


**Sửa:** **đảo** từng bit trên đường chéo.

### Lỗi 4 — Trả về xâu có trong `nums` khi dùng ngẫu nhiên mà không kiểm

Một cách "lười": sinh ngẫu nhiên một xâu và trả về luôn. Xác suất trùng là `n / 2ⁿ` — nhỏ với `n` lớn, nhưng với `n = 1, 2` rất cao (`n = 1`: 50%). **Sửa:** lặp tới khi xâu không có trong `set(nums)`, hoặc dùng cách tất định.

In [23]:
def find_random_unchecked(nums, rnd):
    n = len(nums)
    return ''.join(rnd.choice('01') for _ in range(n))       # SAI: không kiểm

def find_random_checked(nums, rnd):
    n = len(nums); seen = set(nums)
    while True:
        s = ''.join(rnd.choice('01') for _ in range(n))
        if s not in seen:
            return s

rnd = random.Random(7)
fails = sum(not is_valid(["0"], find_random_unchecked(["0"], rnd)) for _ in range(1000))
print("n=1: bản không kiểm sai", fails, "/ 1000 lần | bản có kiểm luôn hợp lệ:", all(is_valid(["0"], find_random_checked(["0"], rnd)) for _ in range(1000)))

n=1: bản không kiểm sai 511 / 1000 lần | bản có kiểm luôn hợp lệ: True


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "`n` đối tượng, tìm một đối tượng **khác tất cả**" ⇒ nghĩ ngay **chuồng bồ câu** (`n + 1` ứng viên) hoặc **chéo hóa** (khác mỗi cái ở một chỗ riêng).
- Nhiều đáp án ⇒ viết `is_valid` và test bằng nó.
- Đổi xâu nhị phân ↔ số: `int(s, 2)`, `format(x, f'0{n}b')` (nhớ đệm 0).
- Khi số đối tượng `m` không bằng `n`: dùng trie/đếm tiền tố để đi xuống nhánh "còn chỗ".
- Liên quan: **LeetCode 268 — Missing Number**, **41 — First Missing Positive** (chuồng bồ câu trên số).

## 15. Biến thể của bài

1. **`m` xâu độ dài `n` với `m < 2ⁿ` bất kỳ:** trie (cách 5).
2. **Tìm xâu thiếu nhỏ nhất theo thứ tự từ điển:** duyệt số `0, 1, 2, …` (cách 2) — dừng trong `m + 1` bước.
3. **Liệt kê mọi xâu thiếu:** `2ⁿ − m` xâu — duyệt toàn bộ hoặc DFS trie.
4. **Bảng chữ cái `k` ký tự:** chéo hóa vẫn đúng: chọn ký tự khác `nums[i][i]`.
5. **LeetCode 1461 — Check If a String Contains All Binary Codes of Size K:** kiểm xem có thiếu mã nào.

In [24]:
def smallest_missing(nums, n):
    seen = {int(s, 2) for s in nums}
    x = 0
    while x in seen:
        x += 1
    return format(x, f'0{n}b')

def diag_k_alphabet(nums, alphabet="abc"):
    return ''.join(next(c for c in alphabet if c != nums[i][i]) for i in range(len(nums)))

def has_all_codes(s, k):
    """LeetCode 1461."""
    return len({s[i:i + k] for i in range(len(s) - k + 1)}) == 2 ** k

print("nhỏ nhất thiếu:", smallest_missing(["000", "001", "011"], 3))
t = ["abc", "bca", "cab"]
print("bảng 3 ký tự:", diag_k_alphabet(t), "| không có trong nums:", diag_k_alphabet(t) not in t)
print("1461 '00110110', k=2 ->", has_all_codes("00110110", 2))

nhỏ nhất thiếu: 010
bảng 3 ký tự: baa | không có trong nums: True
1461 '00110110', k=2 -> True


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Duyệt theo thứ tự | product + set | `O(n²)` thực tế | `O(n²)` | dễ viết |
| 2. Số `0..n` | chuồng bồ câu trên số | `O(n²)` | `O(n)` | cần đáp án nhỏ nhất |
| 3. Đếm bit 1 | chuồng bồ câu trên popcount | `O(n²)` | `O(n)` | mẹo hay |
| 4. Cantor | đảo đường chéo | `O(n)` | `O(1)` | **nộp bài** |
| 5. Trie | đi xuống nhánh còn chỗ | `O(mn)` | `O(mn)` | số xâu `m ≠ n` |
| 6. Ngẫu nhiên có kiểm | sinh tới khi không trùng | kỳ vọng `O(n)` | `O(n²)` | khi cần đáp án ngẫu nhiên |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().findDifferentBinaryString(["1101", "0011", "1110", "0000"])` trả về gì?

**Bài 2 (tính tay):** Với `nums = ["10", "11"]`, liệt kê mọi đáp án hợp lệ và cho biết mỗi cách 2, 3, 4 trả về xâu nào.

**Bài 3 (điền TODO):** Hoàn thiện `all_missing(nums)` trả về **tất cả** xâu độ dài `n` không có trong `nums`, sắp theo thứ tự từ điển.

**Bài 4 (tự viết lại):** Viết lại cách Cantor bằng **phép XOR trên số nguyên**: đổi đường chéo thành số rồi `XOR` với `2ⁿ − 1`.

**Bài 5 (mini challenge):** Cho `m` xâu độ dài `n` (`m < 2ⁿ`, `n ≤ 20`), trả về xâu thiếu **nhỏ nhất theo từ điển** trong `O(m · n)` (không duyệt tuần tự `0, 1, 2, …` nếu `m` lớn) — gợi ý: trie, ưu tiên nhánh `'0'`.

In [25]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def all_missing(nums):
    # TODO: duyệt mọi xâu bằng product('01', repeat=n), lọc theo set
    pass

# Bài 4 — TODO
def cantor_xor(nums):
    # TODO: d = int(đường chéo, 2); return format(d ^ ((1 << n) - 1), f'0{n}b')
    pass

# Bài 5 — TODO
def smallest_missing_trie(nums, n):
    # TODO: như find_trie nhưng luôn thử '0' trước (đã là như vậy) — chứng minh cho ra nhỏ nhất
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Đường chéo: `nums[0][0], nums[1][1], nums[2][2], nums[3][3]`.
- Mức 2: `1, 0, 1, 0`.
- Mức 3: Đảo được `0101`.

**Bài 2**
- Mức 1: Có 4 xâu độ dài 2, hai xâu bị chiếm.
- Mức 2: Đáp án hợp lệ: `00`, `01`.
- Mức 3: Cantor đảo `1, 1` ⇒ `00`.

**Bài 3**
- Mức 1: `product` sinh theo thứ tự từ điển.
- Mức 2: Có `2ⁿ − n` xâu.
- Mức 3: Kiểm số lượng.

**Bài 4**
- Mức 1: Đảo mọi bit = XOR với toàn `1`.
- Mức 2: `(1 << n) − 1`.
- Mức 3: Nhớ đệm 0.

**Bài 5**
- Mức 1: Tại mỗi tầng, nếu nhánh `'0'` còn chỗ thì đi `'0'`.
- Mức 2: Nhánh còn chỗ ⇔ `cnt < 2^(độ sâu còn lại)`.
- Mức 3: So với duyệt `0, 1, 2, …` trên dữ liệu nhỏ.

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1
r1 = Solution().findDifferentBinaryString(["1101", "0011", "1110", "0000"])
print("Bài 1:", r1)
assert r1 == "0101"

# Bài 2
nums2 = ["10", "11"]
print("Bài 2: mọi đáp án:", [''.join(t) for t in product('01', repeat=2) if ''.join(t) not in nums2],
      "| số:", find_small_numbers(nums2), "| bit 1:", find_by_popcount(nums2), "| Cantor:", find_cantor(nums2))

# Bài 3
def all_missing_sol(nums):
    n = len(nums); seen = set(nums)
    return [s for s in (''.join(t) for t in product('01', repeat=n)) if s not in seen]

assert all_missing_sol(["01", "10"]) == ["00", "11"]
assert len(all_missing_sol(["111", "011", "001"])) == 5
print("Bài 3 OK:", all_missing_sol(["111", "011", "001"]))

Bài 1: 0101
Bài 2: mọi đáp án: ['00', '01'] | số: 00 | bit 1: 00 | Cantor: 00
Bài 3 OK: ['000', '010', '100', '101', '110']


In [27]:
# Bài 4
def cantor_xor_sol(nums):
    n = len(nums)
    d = int(''.join(nums[i][i] for i in range(n)), 2)
    return format(d ^ ((1 << n) - 1), f'0{n}b')

rnd = random.Random(4)
for _ in range(2000):
    n = rnd.randint(1, 16)
    t = [format(x, f'0{n}b') for x in rnd.sample(range(2 ** n), n)]
    assert cantor_xor_sol(t) == Solution().findDifferentBinaryString(t)
print("Bài 4 OK")

# Bài 5
def smallest_missing_trie_sol(nums, n):
    return find_trie(nums, n)

for _ in range(2000):
    n = rnd.randint(1, 8)
    m = rnd.randint(0, 2 ** n - 1)
    t = [format(x, f'0{n}b') for x in rnd.sample(range(2 ** n), m)]
    assert smallest_missing_trie_sol(t, n) == smallest_missing(t, n)
print("Bài 5 OK | ví dụ:", smallest_missing_trie_sol(["000", "001", "010"], 3))

Bài 4 OK


Bài 5 OK | ví dụ: 011


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template chéo hóa Cantor:**

```python
return ''.join('1' if nums[i][i] == '0' else '0' for i in range(len(nums)))
```

**Template chuồng bồ câu trên số:**

```python
seen = {int(s, 2) for s in nums}
for x in range(n + 1):            # n + 1 ứng viên
    if x not in seen:
        return format(x, f'0{n}b')    # đệm 0 cho đủ n bit
```

**Checklist:**
- [ ] Đáp án dài đúng `n` (đệm 0)?
- [ ] Đủ `n + 1` ứng viên (chuồng bồ câu)?
- [ ] Chéo hóa có **đảo** bit?
- [ ] Không tin vào ngẫu nhiên mà không kiểm?
- [ ] Đã test bằng `is_valid` (nhiều đáp án)?
- [ ] Chữ ký đúng: `def findDifferentBinaryString(self, nums: List[str]) -> str`?